Objectif : faire une classification à partir des images brutes + segmentation vaisseaux sanguins à partir de notre modèle pour déduire la maladie du patient.

Rappel : 
 - N : normal
 - A : age-related macular degeneration
 - G : glaucoma
 - D : diabetic retinopathy

In [1]:
import os
import sys
import matplotlib.pyplot as plt
import yaml

sys.path.append('..')

In [2]:
# Backbone ResNet25 pre-entraine pour la classification (supporte 1, 3 ou 4 canaux en entree)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models.resnet import ResNet, Bottleneck
from torchvision.models import resnet50, ResNet50_Weights

class ResNet25Classifier(nn.Module):
    def __init__(self, in_channels=3, num_classes=4, pretrained=True):
        super().__init__()

        # ResNet25: profondeur 25 avec bloc Bottleneck -> 1 + 3*(2+2+2+1) + 1 = 25
        self.backbone = ResNet(block=Bottleneck, layers=[2, 2, 2, 1])

        # Adapte la premiere convolution pour gerer 1, 3 ou 4 canaux
        if in_channels != 3:
            self.backbone.conv1 = nn.Conv2d(
                in_channels,
                64,
                kernel_size=7,
                stride=2,
                padding=3,
                bias=False,
            )

        # Tete de classification
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)

        if pretrained:
            # On transfere les poids ImageNet d'un ResNet50 vers les couches compatibles.
            # Les couches inexistantes (ex: blocs de layer4 en moins) sont ignorees.
            base = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
            state = base.state_dict()

            # Conversion conv1 si le nombre de canaux d'entree differe de 3
            if in_channels == 1:
                state['conv1.weight'] = state['conv1.weight'].mean(dim=1, keepdim=True)
            elif in_channels == 4:
                w = state['conv1.weight']
                w4 = torch.zeros((w.size(0), 4, w.size(2), w.size(3)), dtype=w.dtype)
                w4[:, :3] = w
                w4[:, 3:4] = w.mean(dim=1, keepdim=True)
                state['conv1.weight'] = w4

            # On ignore fc (nombre de classes different) et les couches non correspondantes
            state.pop('fc.weight', None)
            state.pop('fc.bias', None)
            self.backbone.load_state_dict(state, strict=False)

    def forward(self, x):
        return self.backbone(x)

## Chargement des données

In [3]:
# Chargement des chemins depuis la config
train_img_path = "../data/train/original/"
train_label_path = "../data/train/ground_truth/"
test_img_path = "../data/test/original/"
test_label_path = "../data/test/ground_truth/"

train_img_files = os.listdir(train_img_path)
train_label_files = os.listdir(train_label_path)
test_img_files = os.listdir(test_img_path)
test_label_files = os.listdir(test_label_path)

train_img = [os.path.join(train_img_path, file) for file in train_img_files]
train_label = [os.path.join(train_label_path, file) for file in train_label_files]
test_img = [os.path.join(test_img_path, file) for file in test_img_files]
test_label = [os.path.join(test_label_path, file) for file in test_label_files]

print(f'Number of training images: {len(train_img)}')
print(f'Number of training labels: {len(train_label)}')
print(f'Number of testing images: {len(test_img)}')
print(f'Number of testing labels: {len(test_label)}')

Number of training images: 600
Number of training labels: 600
Number of testing images: 200
Number of testing labels: 200


In [4]:
import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode
from src.dataset.dataset import VesselDataset
from torch.utils.data import DataLoader, random_split

# Paramètres depuis la config
batch_size = 1
image_size = 512
train_split = 0.8

image_transform = transforms.Compose([
    transforms.Resize((image_size, image_size), interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
])

label_transform = transforms.Compose([
    transforms.Resize((image_size, image_size), interpolation=InterpolationMode.NEAREST),
    transforms.ToTensor(),
])

# Dataset train/val (à partir du dossier train)
dataset = VesselDataset(
    train_img,
    train_label,
    image_transform=image_transform,
    label_transform=label_transform,
 )

train_size = int(train_split * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Dataset test (dossier test)
test_dataset = VesselDataset(
    test_img,
    test_label,
    image_transform=image_transform,
    label_transform=label_transform,
 )
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Dataset créé avec image_size={image_size}, batch_size={batch_size}")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Dataset créé avec image_size=512, batch_size=1
Train: 480, Val: 120, Test: 200


In [5]:
import os
import time
import copy
import torch.optim as optim
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Mapping des classes
label_to_idx = {'N': 0, 'A': 1, 'G': 2, 'D': 3}
idx_to_label = {v: k for k, v in label_to_idx.items()}

def get_safe_device(preferred='cuda'):
    """Choisit un device utilisable. En cas d'erreur CUDA, bascule automatiquement sur CPU."""
    if preferred == 'cuda' and torch.cuda.is_available():
        try:
            _ = torch.zeros(1, device='cuda')
            return torch.device('cuda')
        except Exception as e:
            print(f"CUDA indisponible/instable ({e}). Bascule sur CPU.")
            return torch.device('cpu')
    return torch.device('cpu')

def build_model_input(images, vessel_masks, expected_in_channels):
    """Prépare l'entrée du modèle selon le nombre de canaux attendu.
    - 1 canal: masque vaisseaux seul
    - 3 canaux: image RGB seule
    - 4 canaux: image RGB + masque vaisseaux (1 canal)
    """
    if expected_in_channels == 1:
        return vessel_masks
    if expected_in_channels == 3:
        return images
    if expected_in_channels == 4:
        return torch.cat([images, vessel_masks], dim=1)
    raise ValueError(f"in_channels={expected_in_channels} non supporté (attendu: 1, 3 ou 4).")

def encode_labels(disease_letters, label_map):
    cleaned = [str(d).strip().upper() for d in disease_letters]
    unknown = [d for d in cleaned if d not in label_map]
    if unknown:
        raise ValueError(f"Labels inconnus trouvés: {unknown}")
    return torch.tensor([label_map[d] for d in cleaned], dtype=torch.long)

def validate_targets(targets, num_classes):
    tmin = int(targets.min().item())
    tmax = int(targets.max().item())
    if tmin < 0 or tmax >= num_classes:
        raise ValueError(
            f"Targets hors plage pour CrossEntropyLoss: min={tmin}, max={tmax}, "
            f"num_classes={num_classes}"
        )

def load_classification_checkpoint(checkpoint_path, model, optimizer=None, device='cpu'):
    """Charge un checkpoint de classification pour reprendre l'entraînement."""
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint introuvable: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])

    if optimizer is not None and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    history = checkpoint.get('history', {})
    history = {
        'train_loss': history.get('train_loss', []),
        'train_acc': history.get('train_acc', []),
        'val_loss': history.get('val_loss', []),
        'val_acc': history.get('val_acc', []),
    }
    start_epoch = int(checkpoint.get('epoch', len(history['train_loss'])))

    print(f"Checkpoint chargé depuis: {checkpoint_path}")
    print(f"Epoch reprise: {start_epoch}")
    return model, optimizer, history, start_epoch

def train_classifier(
    model,
    train_loader,
    val_loader,
    n_epochs=10,
    learning_rate=1e-3,
    device=None,
    save_root_dir='../saved_models',
    expected_in_channels=3,
    num_classes=None,
    model_dir=None,
    optimizer=None,
    initial_history=None,
    start_epoch=0,
    ):
    """Entraîne un classifieur multi-classes, sauvegarde best model + checkpoint, et retourne modèle + historique."""
    if num_classes is None:
        if hasattr(model, 'classifier') and hasattr(model.classifier, 'out_features'):
            num_classes = model.classifier.out_features
        else:
            raise ValueError("Impossible d'inférer num_classes. Passe num_classes explicitement.")

    if device is None:
        device = get_safe_device('cuda')

    try:
        model = model.to(device)
    except Exception as e:
        print(f"Échec sur {device} ({e}). Retry sur CPU.")
        device = torch.device('cpu')
        model = model.to(device)

    if model_dir is None:
        run_name = f"classification_train_{time.strftime('%Y%m%d-%H%M%S')}"
        model_dir = os.path.join(save_root_dir, run_name)
    os.makedirs(model_dir, exist_ok=True)
    best_model_path = os.path.join(model_dir, 'best_model.pth')
    checkpoint_path = os.path.join(model_dir, 'checkpoint.pth')

    if optimizer is None:
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    if initial_history is None:
        history = {
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': [],
        }
    else:
        history = {
            'train_loss': list(initial_history.get('train_loss', [])),
            'train_acc': list(initial_history.get('train_acc', [])),
            'val_loss': list(initial_history.get('val_loss', [])),
            'val_acc': list(initial_history.get('val_acc', [])),
        }

    if start_epoch == 0 and len(history['train_loss']) > 0:
        start_epoch = len(history['train_loss'])

    best_val_acc = max(history['val_acc']) if len(history['val_acc']) > 0 else 0.0
    best_epoch = (history['val_acc'].index(best_val_acc) + 1) if len(history['val_acc']) > 0 else 0
    best_state = copy.deepcopy(model.state_dict())

    if start_epoch >= n_epochs:
        print(f"Aucun entraînement supplémentaire: start_epoch={start_epoch} >= n_epochs={n_epochs}")
        return model, history, model_dir

    criterion = nn.CrossEntropyLoss()

    for epoch in range(start_epoch + 1, n_epochs + 1):
        # --- Training ---
        model.train()
        running_loss, running_correct, running_total = 0.0, 0, 0

        train_iter = tqdm(
            train_loader,
            desc=f"Epoch {epoch:02d}/{n_epochs} [train]",
            leave=False,
            dynamic_ncols=True
        )
        for images, vessel_masks, disease_letters in train_iter:
            targets = encode_labels(disease_letters, label_to_idx)
            validate_targets(targets, num_classes)

            images = images.to(device)
            vessel_masks = vessel_masks.to(device)
            targets = targets.to(device)

            inputs = build_model_input(images, vessel_masks, expected_in_channels)

            optimizer.zero_grad()
            logits = model(inputs)
            if logits.shape[1] != num_classes:
                raise ValueError(
                    f"Sortie modèle incohérente: logits.shape[1]={logits.shape[1]} != num_classes={num_classes}"
                )

            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()

            preds = torch.argmax(logits, dim=1)
            bs = targets.size(0)
            running_loss += loss.item() * bs
            running_correct += (preds == targets).sum().item()
            running_total += bs

            train_iter.set_postfix(
                loss=f"{(running_loss / max(running_total, 1)):.4f}",
                acc=f"{(running_correct / max(running_total, 1)):.4f}"
            )

        train_loss = running_loss / max(running_total, 1)
        train_acc = running_correct / max(running_total, 1)

        # --- Validation ---
        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0

        val_iter = tqdm(
            val_loader,
            desc=f"Epoch {epoch:02d}/{n_epochs} [val]",
            leave=False,
            dynamic_ncols=True
        )
        with torch.no_grad():
            for images, vessel_masks, disease_letters in val_iter:
                targets = encode_labels(disease_letters, label_to_idx)
                validate_targets(targets, num_classes)

                images = images.to(device)
                vessel_masks = vessel_masks.to(device)
                targets = targets.to(device)

                inputs = build_model_input(images, vessel_masks, expected_in_channels)
                logits = model(inputs)
                if logits.shape[1] != num_classes:
                    raise ValueError(
                        f"Sortie modèle incohérente: logits.shape[1]={logits.shape[1]} != num_classes={num_classes}"
                    )

                loss = criterion(logits, targets)

                preds = torch.argmax(logits, dim=1)
                bs = targets.size(0)
                val_loss_sum += loss.item() * bs
                val_correct += (preds == targets).sum().item()
                val_total += bs

                val_iter.set_postfix(
                    loss=f"{(val_loss_sum / max(val_total, 1)):.4f}",
                    acc=f"{(val_correct / max(val_total, 1)):.4f}"
                )

        val_loss = val_loss_sum / max(val_total, 1)
        val_acc = val_correct / max(val_total, 1)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Sauvegarde du meilleur modèle
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, best_model_path)

        # Sauvegarde du checkpoint complet à chaque epoch
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history,
            'best_val_acc': best_val_acc,
            'best_epoch': best_epoch,
            'num_classes': num_classes,
            'expected_in_channels': expected_in_channels,
            'label_to_idx': label_to_idx,
        }
        torch.save(checkpoint, checkpoint_path)

        print(
            f"Epoch {epoch:02d}/{n_epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

    model.load_state_dict(best_state)
    print(f"Meilleure val_acc: {best_val_acc:.4f} (epoch {best_epoch})")
    print(f"Best model: {best_model_path}")
    print(f"Checkpoint: {checkpoint_path}")
    return model, history, model_dir

def predict_and_evaluate(
    model,
    data_loader,
    expected_in_channels=3,
    device=None,
    label_map=None,
    ):
    """Prédit sur un DataLoader et retourne des métriques globales et par classe."""
    if label_map is None:
        label_map = label_to_idx
    inv_map = {v: k for k, v in label_map.items()}

    if device is None:
        device = get_safe_device('cuda')

    model = model.to(device)
    model.eval()

    all_targets = []
    all_preds = []

    with torch.no_grad():
        test_iter = tqdm(data_loader, desc='Test [predict]', leave=False, dynamic_ncols=True)
        for images, vessel_masks, disease_letters in test_iter:
            targets = encode_labels(disease_letters, label_map)

            images = images.to(device)
            vessel_masks = vessel_masks.to(device)
            inputs = build_model_input(images, vessel_masks, expected_in_channels)

            logits = model(inputs)
            preds = torch.argmax(logits, dim=1).detach().cpu()

            all_targets.extend(targets.tolist())
            all_preds.extend(preds.tolist())

    labels_sorted = sorted(inv_map.keys())

    accuracy_global = accuracy_score(all_targets, all_preds)
    precision_global, recall_global, f1_global, _ = precision_recall_fscore_support(
        all_targets,
        all_preds,
        labels=labels_sorted,
        average='weighted',
        zero_division=0,
    )

    precision_cls, recall_cls, f1_cls, support_cls = precision_recall_fscore_support(
        all_targets,
        all_preds,
        labels=labels_sorted,
        average=None,
        zero_division=0,
    )

    metrics_by_class = {}
    for i, cls_idx in enumerate(labels_sorted):
        cls_name = inv_map[cls_idx]
        metrics_by_class[cls_name] = {
            'precision': float(precision_cls[i]),
            'recall': float(recall_cls[i]),
            'f1': float(f1_cls[i]),
            'support': int(support_cls[i]),
        }

    metrics_global = {
        'accuracy': float(accuracy_global),
        'precision_weighted': float(precision_global),
        'recall_weighted': float(recall_global),
        'f1_weighted': float(f1_global),
    }

    return {
        'global': metrics_global,
        'by_class': metrics_by_class,
        'y_true': all_targets,
        'y_pred': all_preds,
    }

/home/msaunier/Documents/ITI5/MIA/MIA-eyes-vessel-segmentation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config img seule

In [6]:
# Chargement de la configuration
config_path = '../configs/config_img_classification.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Configuration chargée :")
print(f"| Modèle chargé : {config['cnn']['name']}")
print(f"| Canaux d'entrée: {config['cnn']['in_channels']}")
print(f"| Canaux de sortie: {config['cnn']['out_channels']}")
print(f"| Epochs: {config['training']['n_epochs']}")
print(f"| Learning rate: {config['training']['learning_rate']}")
print(f"| Resume training: {config['training']['resume_training']}")

Configuration chargée :
| Modèle chargé : SimpleCNN
| Canaux d'entrée: 3
| Canaux de sortie: 4
| Epochs: 30
| Learning rate: 0.0001
| Resume training: False


In [7]:
# --------------------------
# Entrainement + sauvegarde (+ reprise possible)
# --------------------------
device = get_safe_device('cuda')
print(f"Device utilisé: {device}")
in_channels = config['cnn']['in_channels']
num_classes = config['cnn']['out_channels']

model = ResNet25Classifier(in_channels=in_channels, num_classes=num_classes, pretrained=True)
optimizer = optim.Adam(model.parameters(), lr=config['training']['learning_rate'])

# Variables pour la reprise
initial_history = None
start_epoch = 0
model_dir = '../saved_models/classification_train_img'

if config['training'].get('resume_training', False):
    checkpoint_path = config['training'].get('resume_checkpoint_path')
    if checkpoint_path and os.path.isfile(checkpoint_path):
        try:
            model, optimizer, initial_history, start_epoch = load_classification_checkpoint(
                checkpoint_path=checkpoint_path,
                model=model,
                optimizer=optimizer,
                device=device,
            )
            model_dir = os.path.dirname(checkpoint_path)
            print("\n✓ Reprise activée avec succès")
            print(f"  Historique chargé: {len(initial_history['train_loss'])} epochs")
        except Exception as e:
            print(f"\n⚠ Erreur lors du chargement du checkpoint: {e}")
            print("  Démarrage d'un nouvel entraînement...")
    else:
        print("\n⚠ Chemin du checkpoint non spécifié ou fichier introuvable")
        print("  Démarrage d'un nouvel entraînement...")
else:
    print("\n  Démarrage d'un nouvel entraînement...")

trained_model, history, model_dir = train_classifier(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=config['training']['n_epochs'],
    learning_rate=config['training']['learning_rate'],
    device=device,
    save_root_dir=config['paths']['save_dir'],
    expected_in_channels=in_channels,
    num_classes=num_classes,
    model_dir=model_dir,
    optimizer=optimizer,
    initial_history=initial_history,
    start_epoch=start_epoch,
    )

print(f"Dossier de sauvegarde: {model_dir}")

Device utilisé: cuda
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/msaunier/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:09<00:00, 10.9MB/s]



  Démarrage d'un nouvel entraînement...


Epoch 01/30 | train_loss=0.8047 train_acc=0.6667 | val_loss=2.6062 val_acc=0.2750


Epoch 02/30 | train_loss=0.4572 train_acc=0.8125 | val_loss=4.3561 val_acc=0.4583


...

Epoch 26/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=5.8672 val_acc=0.6000


Epoch 27/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=7.8437 val_acc=0.4833


Epoch 28/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=9.1449 val_acc=0.3833


Epoch 29/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=18.1137 val_acc=0.4917


Epoch 30/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=11.1424 val_acc=0.4500
Meilleure val_acc: 0.6667 (epoch 15)
Best model: ../saved_models/classification_train_img/best_model.pth
Checkpoint: ../saved_models/classification_train_img/checkpoint.pth
Dossier de sauvegarde: ../saved_models/classification_train_img


In [8]:
# Chargement du meilleur modèle (optionnel, pour démontrer la réutilisation)
trained_model = ResNet25Classifier(in_channels=in_channels, num_classes=num_classes, pretrained=False)

best_model_path = os.path.join(model_dir, 'best_model.pth')
if os.path.exists(best_model_path):
    print(f"Chargement du meilleur modèle depuis: {best_model_path}")
    best_state = torch.load(best_model_path, map_location=device)
    trained_model.load_state_dict(best_state)
else:
    print(f"Aucun meilleur modèle trouvé à: {best_model_path}. Utilisation du modèle entraîné actuel.")

# --------------------------
# Prédiction sur jeu de test + métriques détaillées
# --------------------------
test_results = predict_and_evaluate(
    model=trained_model,
    data_loader=test_loader,
    expected_in_channels=in_channels,
    device=device,
    label_map=label_to_idx,
    )

print("\n=== Métriques globales ===")
for metric_name, metric_value in test_results['global'].items():
    print(f"{metric_name}: {metric_value:.6f}")

print("\n=== Métriques par classe ===")
for cls_name, cls_metrics in test_results['by_class'].items():
    print(f"Classe {cls_name}:")
    print(f"  precision: {cls_metrics['precision']:.6f}")
    print(f"  recall:    {cls_metrics['recall']:.6f}")
    print(f"  f1:        {cls_metrics['f1']:.6f}")
    print(f"  support:   {cls_metrics['support']}")

Chargement du meilleur modèle depuis: ../saved_models/classification_train_img/best_model.pth



=== Métriques globales ===
accuracy: 0.420000
precision_weighted: 0.365383
recall_weighted: 0.420000
f1_weighted: 0.383000

=== Métriques par classe ===
Classe N:
  precision: 0.105263
  recall:    0.040000
  f1:        0.057971
  support:   50
Classe A:
  precision: 0.519231
  recall:    0.540000
  f1:        0.529412
  support:   50
Classe G:
  precision: 0.466667
  recall:    0.700000
  f1:        0.560000
  support:   50
Classe D:
  precision: 0.370370
  recall:    0.400000
  f1:        0.384615
  support:   50


## Image et mask

In [9]:
# Chargement de la configuration
config_path = '../configs/config_img_mask_classification.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Configuration chargée :")
print(f"| Modèle chargé : {config['cnn']['name']}")
print(f"| Canaux d'entrée: {config['cnn']['in_channels']}")
print(f"| Canaux de sortie: {config['cnn']['out_channels']}")
print(f"| Epochs: {config['training']['n_epochs']}")
print(f"| Learning rate: {config['training']['learning_rate']}")
print(f"| Resume training: {config['training']['resume_training']}")

Configuration chargée :
| Modèle chargé : SimpleCNN
| Canaux d'entrée: 4
| Canaux de sortie: 4
| Epochs: 30
| Learning rate: 0.0001
| Resume training: False


In [10]:
# --------------------------
# Entrainement + sauvegarde (+ reprise possible)
# --------------------------
device = get_safe_device('cuda')
print(f"Device utilisé: {device}")
in_channels = config['cnn']['in_channels']
num_classes = config['cnn']['out_channels']

model = ResNet25Classifier(in_channels=in_channels, num_classes=num_classes, pretrained=True)
optimizer = optim.Adam(model.parameters(), lr=config['training']['learning_rate'])

# Variables pour la reprise
initial_history = None
start_epoch = 0
model_dir = '../saved_models/classification_train_img_mask'

if config['training'].get('resume_training', False):
    checkpoint_path = config['training'].get('resume_checkpoint_path')
    if checkpoint_path and os.path.isfile(checkpoint_path):
        try:
            model, optimizer, initial_history, start_epoch = load_classification_checkpoint(
                checkpoint_path=checkpoint_path,
                model=model,
                optimizer=optimizer,
                device=device,
            )
            model_dir = os.path.dirname(checkpoint_path)
            print("\n✓ Reprise activée avec succès")
            print(f"  Historique chargé: {len(initial_history['train_loss'])} epochs")
        except Exception as e:
            print(f"\n⚠ Erreur lors du chargement du checkpoint: {e}")
            print("  Démarrage d'un nouvel entraînement...")
    else:
        print("\n⚠ Chemin du checkpoint non spécifié ou fichier introuvable")
        print("  Démarrage d'un nouvel entraînement...")
else:
    print("\n  Démarrage d'un nouvel entraînement...")

trained_model, history, model_dir = train_classifier(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=config['training']['n_epochs'],
    learning_rate=config['training']['learning_rate'],
    device=device,
    save_root_dir=config['paths']['save_dir'],
    expected_in_channels=in_channels,
    num_classes=num_classes,
    model_dir=model_dir,
    optimizer=optimizer,
    initial_history=initial_history,
    start_epoch=start_epoch,
    )

print(f"Dossier de sauvegarde: {model_dir}")

Device utilisé: cuda

  Démarrage d'un nouvel entraînement...


Epoch 01/30 | train_loss=0.9223 train_acc=0.6188 | val_loss=1.1016 val_acc=0.6750


Epoch 02/30 | train_loss=0.4387 train_acc=0.8500 | val_loss=2.7375 val_acc=0.5917


Epoch 03/30 | train_loss=0.2922 train_acc=0.8896 | val_loss=1.7570 val_acc=0.5917


Epoch 04/30 | train_loss=0.1670 train_acc=0.9563 | val_loss=2.2331 val_acc=0.5250


...

Epoch 26/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=5.5150 val_acc=0.6583


Epoch 27/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=42.3725 val_acc=0.5583


Epoch 28/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=20.1346 val_acc=0.4583


Epoch 29/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=9.4121 val_acc=0.5750


Epoch 30/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=4.1735 val_acc=0.6417
Meilleure val_acc: 0.7083 (epoch 16)
Best model: ../saved_models/classification_train_img_mask/best_model.pth
Checkpoint: ../saved_models/classification_train_img_mask/checkpoint.pth
Dossier de sauvegarde: ../saved_models/classification_train_img_mask


In [11]:
# Chargement du meilleur modèle (optionnel, pour démontrer la réutilisation)
trained_model = ResNet25Classifier(in_channels=in_channels, num_classes=num_classes, pretrained=False)

best_model_path = os.path.join(model_dir, 'best_model.pth')
if os.path.exists(best_model_path):
    print(f"Chargement du meilleur modèle depuis: {best_model_path}")
    best_state = torch.load(best_model_path, map_location=device)
    trained_model.load_state_dict(best_state)
else:
    print(f"Aucun meilleur modèle trouvé à: {best_model_path}. Utilisation du modèle entraîné actuel.")

# --------------------------
# Prédiction sur jeu de test + métriques détaillées
# --------------------------
test_results = predict_and_evaluate(
    model=trained_model,
    data_loader=test_loader,
    expected_in_channels=in_channels,
    device=device,
    label_map=label_to_idx,
    )

print("\n=== Métriques globales ===")
for metric_name, metric_value in test_results['global'].items():
    print(f"{metric_name}: {metric_value:.6f}")

print("\n=== Métriques par classe ===")
for cls_name, cls_metrics in test_results['by_class'].items():
    print(f"Classe {cls_name}:")
    print(f"  precision: {cls_metrics['precision']:.6f}")
    print(f"  recall:    {cls_metrics['recall']:.6f}")
    print(f"  f1:        {cls_metrics['f1']:.6f}")
    print(f"  support:   {cls_metrics['support']}")

Chargement du meilleur modèle depuis: ../saved_models/classification_train_img_mask/best_model.pth



=== Métriques globales ===
accuracy: 0.445000
precision_weighted: 0.584842
recall_weighted: 0.445000
f1_weighted: 0.392082

=== Métriques par classe ===
Classe N:
  precision: 1.000000
  recall:    0.040000
  f1:        0.076923
  support:   50
Classe A:
  precision: 0.363636
  recall:    0.560000
  f1:        0.440945
  support:   50
Classe G:
  precision: 0.486842
  recall:    0.740000
  f1:        0.587302
  support:   50
Classe D:
  precision: 0.488889
  recall:    0.440000
  f1:        0.463158
  support:   50


## Mask seul

In [12]:
# Chargement de la configuration
config_path = '../configs/config_mask_classification.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Configuration chargée :")
print(f"| Modèle chargé : {config['cnn']['name']}")
print(f"| Canaux d'entrée: {config['cnn']['in_channels']}")
print(f"| Canaux de sortie: {config['cnn']['out_channels']}")
print(f"| Epochs: {config['training']['n_epochs']}")
print(f"| Learning rate: {config['training']['learning_rate']}")
print(f"| Resume training: {config['training']['resume_training']}")

Configuration chargée :
| Modèle chargé : SimpleCNN
| Canaux d'entrée: 1
| Canaux de sortie: 4
| Epochs: 30
| Learning rate: 0.0001
| Resume training: False


In [13]:
# --------------------------
# Entrainement + sauvegarde (+ reprise possible)
# --------------------------
device = get_safe_device('cuda')
print(f"Device utilisé: {device}")
in_channels = config['cnn']['in_channels']
num_classes = config['cnn']['out_channels']

model = ResNet25Classifier(in_channels=in_channels, num_classes=num_classes, pretrained=True)
optimizer = optim.Adam(model.parameters(), lr=config['training']['learning_rate'])

# Variables pour la reprise
initial_history = None
start_epoch = 0
model_dir = '../saved_models/classification_train_mask'

if config['training'].get('resume_training', False):
    checkpoint_path = config['training'].get('resume_checkpoint_path')
    if checkpoint_path and os.path.isfile(checkpoint_path):
        try:
            model, optimizer, initial_history, start_epoch = load_classification_checkpoint(
                checkpoint_path=checkpoint_path,
                model=model,
                optimizer=optimizer,
                device=device,
            )
            model_dir = os.path.dirname(checkpoint_path)
            print("\n✓ Reprise activée avec succès")
            print(f"  Historique chargé: {len(initial_history['train_loss'])} epochs")
        except Exception as e:
            print(f"\n⚠ Erreur lors du chargement du checkpoint: {e}")
            print("  Démarrage d'un nouvel entraînement...")
    else:
        print("\n⚠ Chemin du checkpoint non spécifié ou fichier introuvable")
        print("  Démarrage d'un nouvel entraînement...")
else:
    print("\n  Démarrage d'un nouvel entraînement...")

trained_model, history, model_dir = train_classifier(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=config['training']['n_epochs'],
    learning_rate=config['training']['learning_rate'],
    device=device,
    save_root_dir=config['paths']['save_dir'],
    expected_in_channels=in_channels,
    num_classes=num_classes,
    model_dir=model_dir,
    optimizer=optimizer,
    initial_history=initial_history,
    start_epoch=start_epoch,
    )

print(f"Dossier de sauvegarde: {model_dir}")

Device utilisé: cuda

  Démarrage d'un nouvel entraînement...


Epoch 01/30 | train_loss=1.3290 train_acc=0.3479 | val_loss=1.5116 val_acc=0.1917


Epoch 02/30 | train_loss=1.2075 train_acc=0.4271 | val_loss=1.8407 val_acc=0.2083


Epoch 03/30 | train_loss=1.0452 train_acc=0.5333 | val_loss=1.4160 val_acc=0.3167


Epoch 04/30 | train_loss=0.7961 train_acc=0.7250 | val_loss=1.4762 val_acc=0.2500


...

Epoch 26/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=2.1438 val_acc=0.2833


Epoch 27/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=2.1329 val_acc=0.2750


Epoch 28/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=2.2029 val_acc=0.2667


Epoch 29/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=2.8653 val_acc=0.2833


Epoch 30/30 | train_loss=0.0000 train_acc=1.0000 | val_loss=2.4358 val_acc=0.2417
Meilleure val_acc: 0.3500 (epoch 10)
Best model: ../saved_models/classification_train_mask/best_model.pth
Checkpoint: ../saved_models/classification_train_mask/checkpoint.pth
Dossier de sauvegarde: ../saved_models/classification_train_mask


In [14]:
# Chargement du meilleur modèle (optionnel, pour démontrer la réutilisation)
trained_model = ResNet25Classifier(in_channels=in_channels, num_classes=num_classes, pretrained=False)

best_model_path = os.path.join(model_dir, 'best_model.pth')
if os.path.exists(best_model_path):
    print(f"Chargement du meilleur modèle depuis: {best_model_path}")
    best_state = torch.load(best_model_path, map_location=device)
    trained_model.load_state_dict(best_state)
else:
    print(f"Aucun meilleur modèle trouvé à: {best_model_path}. Utilisation du modèle entraîné actuel.")

# --------------------------
# Prédiction sur jeu de test + métriques détaillées
# --------------------------
test_results = predict_and_evaluate(
    model=trained_model,
    data_loader=test_loader,
    expected_in_channels=in_channels,
    device=device,
    label_map=label_to_idx,
    )

print("\n=== Métriques globales ===")
for metric_name, metric_value in test_results['global'].items():
    print(f"{metric_name}: {metric_value:.6f}")

print("\n=== Métriques par classe ===")
for cls_name, cls_metrics in test_results['by_class'].items():
    print(f"Classe {cls_name}:")
    print(f"  precision: {cls_metrics['precision']:.6f}")
    print(f"  recall:    {cls_metrics['recall']:.6f}")
    print(f"  f1:        {cls_metrics['f1']:.6f}")
    print(f"  support:   {cls_metrics['support']}")

Chargement du meilleur modèle depuis: ../saved_models/classification_train_mask/best_model.pth



=== Métriques globales ===
accuracy: 0.345000
precision_weighted: 0.294902
recall_weighted: 0.345000
f1_weighted: 0.281193

=== Métriques par classe ===
Classe N:
  precision: 0.000000
  recall:    0.000000
  f1:        0.000000
  support:   50
Classe A:
  precision: 0.416667
  recall:    0.500000
  f1:        0.454545
  support:   50
Classe G:
  precision: 0.473684
  recall:    0.180000
  f1:        0.260870
  support:   50
Classe D:
  precision: 0.289256
  recall:    0.700000
  f1:        0.409357
  support:   50
